In [1]:
import os

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.getenv("HF_TOKEN")

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
    print("HF token loaded from Colab Secret/environment.")
else:
    print("HF token not configured; public models may still work.")


HF token loaded from Kaggle Secrets.


In [2]:
!git clone https://github.com/TiiAyyLuvBear/Text-Mining---RAG-on-News

Cloning into 'Text-Mining---RAG-on-News'...
remote: Enumerating objects: 645, done.
remote: Counting objects: 100% (301/301), done.
remote: Compressing objects: 100% (157/157), done.
remote: Total 645 (delta 177), reused 259 (delta 144), pack-reused 344 (from 1)
Receiving objects: 100% (645/645), 22.11 MiB | 20.94 MiB/s, done.
Resolving deltas: 100% (351/351), done.


In [3]:
%cd Text-Mining---RAG-on-News
!git checkout test_feature_ta
!git fetch
!git pull
!pip install -r requirements.txt

/content/Text-Mining---RAG-on-News
Branch 'test_feature_ta' set up to track remote branch 'test_feature_ta' from 'origin'.
Switched to a new branch 'test_feature_ta'
Already up to date.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 87.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 123.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 122.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 100.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 21.3 MB/s 

In [4]:
%cd Text-Mining---RAG-on-News

[Errno 2] No such file or directory: 'Text-Mining---RAG-on-News'
/content/Text-Mining---RAG-on-News


In [5]:
from pathlib import Path
import json
import subprocess
import sys
import time

import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Project root:", PROJECT_ROOT)

Project root: /content/Text-Mining---RAG-on-News


## 1. Cấu hình

- `RAW_CSV_PATH`: CSV đầu vào cho `src.data_ingestion.cli`.
- `ARTICLE_LIMIT`: số article đưa vào chunking/embedding. Đặt `None` để chạy full corpus.
- `STRATEGIES`: các chiến thuật chunking cần embed.
- `BATCH_SIZE`: CPU nên bắt đầu thấp, ví dụ 4 hoặc 8.

In [6]:
# Chọn một CSV đang có trong repo. Có thể đổi sang validation_new.csv/test_new.csv hoặc CSV tổng của bạn.
RAW_CSV_PATH = PROJECT_ROOT / "Dataset" / "Create_QA_Vietonline" / "VietOnlineNews" / "train_new.csv"

PROCESSED_JSONL = PROJECT_ROOT / "src" / "embed" / "output" / "data" / "vieonline_news_clean.jsonl"
REVIEW_JSONL = PROJECT_ROOT / "src" / "embed" / "output" / "data" / "vieonline_news_human_review.jsonl"
CHUNK_OUTPUT_DIR = PROJECT_ROOT / "src" / "embed" / "output" / "chunks"
DENSE_OUTPUT_ROOT = PROJECT_ROOT / "src" / "embed" / "output" / "dense"
SAMPLE_REPORT = PROJECT_ROOT / "src" / "embed" / "output" / "embedding_strategy_samples.md"

# CPU-friendly default. Đặt None để chạy toàn bộ file sau khi đã smoke test.
ARTICLE_LIMIT = None
STRATEGIES = ["token", "llamaindex", "structured", "langchain_recursive"]  # thêm "langchain_recursive" nếu muốn so sánh đủ 4 strategy

CHUNK_SIZE = 400
OVERLAP = 80
SMALL_ARTICLE_CHARS = 1000
BATCH_SIZE = 4
MODEL_NAME = "intfloat/multilingual-e5-large"
SAMPLE_SIZE = 5

for path in [PROCESSED_JSONL.parent, CHUNK_OUTPUT_DIR, DENSE_OUTPUT_ROOT, SAMPLE_REPORT.parent]:
    path.mkdir(parents=True, exist_ok=True)

config = {
    "raw_csv": str(RAW_CSV_PATH),
    "processed_jsonl": str(PROCESSED_JSONL),
    "chunk_output_dir": str(CHUNK_OUTPUT_DIR),
    "dense_output_root": str(DENSE_OUTPUT_ROOT),
    "article_limit": ARTICLE_LIMIT,
    "strategies": STRATEGIES,
    "chunk_size": CHUNK_SIZE,
    "overlap": OVERLAP,
    "batch_size": BATCH_SIZE,
    "model_name": MODEL_NAME,
}
display(config)

{'raw_csv': '/content/Text-Mining---RAG-on-News/Dataset/Create_QA_Vietonline/VietOnlineNews/train_new.csv',
 'processed_jsonl': '/content/Text-Mining---RAG-on-News/src/embed/output/data/vieonline_news_clean.jsonl',
 'chunk_output_dir': '/content/Text-Mining---RAG-on-News/src/embed/output/chunks',
 'dense_output_root': '/content/Text-Mining---RAG-on-News/src/embed/output/dense',
 'article_limit': None,
 'strategies': ['token', 'llamaindex', 'structured', 'langchain_recursive'],
 'chunk_size': 400,
 'overlap': 80,
 'batch_size': 4,
 'model_name': 'intfloat/multilingual-e5-large'}

## 2. Helper chạy CLI có streaming output

Các lệnh bên dưới dùng `sys.executable -m ...` để chạy đúng Python environment của notebook. Output được stream trực tiếp để thấy progress bar của chunking và embedding.

In [7]:
def run_cli(args, cwd=PROJECT_ROOT):
    args = [str(item) for item in args]
    printable = " ".join(args)
    print(f"\n$ {printable}\n")
    started = time.perf_counter()
    process = subprocess.Popen(
        args,
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        lines.append(line)
    return_code = process.wait()
    elapsed = time.perf_counter() - started
    print(f"\n[exit={return_code}] elapsed={elapsed:.2f}s")
    if return_code != 0:
        raise RuntimeError(f"Command failed: {printable}")
    return "".join(lines)

## 3. Data ingestion

In [8]:
ingest_cmd = [
    sys.executable, "-m", "src.data_ingestion.cli",
    "--input", RAW_CSV_PATH,
    "--output", PROCESSED_JSONL,
    "--review-output", REVIEW_JSONL,
]
run_cli(ingest_cmd)


$ /usr/bin/python3 -m src.data_ingestion.cli --input /content/Text-Mining---RAG-on-News/Dataset/Create_QA_Vietonline/VietOnlineNews/train_new.csv --output /content/Text-Mining---RAG-on-News/src/embed/output/data/vieonline_news_clean.jsonl --review-output /content/Text-Mining---RAG-on-News/src/embed/output/data/vieonline_news_human_review.jsonl

{
  "rows_read": 10073,
  "rows_written": 10073,
  "missing_required": 0,
  "short_content": 13,
  "long_content": 3,
  "html_like": 3,
  "url_like": 0
}

[exit=0] elapsed=32.27s


'{\n  "rows_read": 10073,\n  "rows_written": 10073,\n  "missing_required": 0,\n  "short_content": 13,\n  "long_content": 3,\n  "html_like": 3,\n  "url_like": 0\n}\n'

## 4. Chunking theo strategy

Bước này dùng CLI `src.chunking.cli`. Nếu chạy CPU, giữ `ARTICLE_LIMIT` nhỏ trước để giảm thời gian embed sau đó.

In [9]:
chunk_cmd = [
    sys.executable, "-m", "src.chunking.cli",
    "--input", PROCESSED_JSONL,
    "--output-dir", CHUNK_OUTPUT_DIR,
    "--strategies", *STRATEGIES,
    "--chunk-size", CHUNK_SIZE,
    "--overlap", OVERLAP,
    "--small-article-chars", SMALL_ARTICLE_CHARS,
]
if ARTICLE_LIMIT is not None:
    chunk_cmd += ["--limit", ARTICLE_LIMIT]
run_cli(chunk_cmd)


$ /usr/bin/python3 -m src.chunking.cli --input /content/Text-Mining---RAG-on-News/src/embed/output/data/vieonline_news_clean.jsonl --output-dir /content/Text-Mining---RAG-on-News/src/embed/output/chunks --strategies token llamaindex structured langchain_recursive --chunk-size 400 --overlap 80 --small-article-chars 1000

Starting chunking: 10073 article(s), 4 strategy(ies), output_dir=/content/Text-Mining---RAG-on-News/src/embed/output/chunks

Chunking [token]: 100%|██████████| 10073/10073 [00:05<00:00, 1866.14article/s]
Completed token: 10073 article(s), 24099 chunk(s), 5.39781s -> /content/Text-Mining---RAG-on-News/src/embed/output/chunks/vieonline_news_chunks_token.jsonl
{
  "token": {
    "output_path": "/content/Text-Mining---RAG-on-News/src/embed/output/chunks/vieonline_news_chunks_token.jsonl",
    "config": {
      "strategy": "token",
      "chunk_size": 400,
      "overlap": 80,
      "min_chunk_tokens": 80,
      "small_article_chars": 1000,
      "max_chunks_per_article": n

'Starting chunking: 10073 article(s), 4 strategy(ies), output_dir=/content/Text-Mining---RAG-on-News/src/embed/output/chunks\n\nChunking [token]:   0%|          | 0/10073 [00:00<?, ?article/s]\nChunking [token]:   1%|          | 121/10073 [00:00<00:08, 1200.61article/s]\nChunking [token]:   2%|▏         | 242/10073 [00:00<00:08, 1093.33article/s]\nChunking [token]:   4%|▎         | 363/10073 [00:00<00:08, 1140.54article/s]\nChunking [token]:   5%|▍         | 494/10073 [00:00<00:07, 1199.17article/s]\nChunking [token]:   6%|▌         | 615/10073 [00:00<00:08, 1158.67article/s]\nChunking [token]:   7%|▋         | 744/10073 [00:00<00:07, 1196.78article/s]\nChunking [token]:   9%|▊         | 866/10073 [00:00<00:07, 1203.64article/s]\nChunking [token]:  10%|▉         | 987/10073 [00:00<00:07, 1198.63article/s]\nChunking [token]:  11%|█         | 1123/10073 [00:00<00:07, 1246.44article/s]\nChunking [token]:  12%|█▏        | 1248/10073 [00:01<00:07, 1188.55article/s]\nChunking [token]:  14%|█

## 5. Xem mẫu khác biệt input embedding giữa các strategy

In [10]:
chunk_files = [CHUNK_OUTPUT_DIR / f"vieonline_news_chunks_{strategy}.jsonl" for strategy in STRATEGIES]
compare_cmd = [
    sys.executable, "-m", "src.embed.compare_embedding_inputs",
    "--inputs", *chunk_files,
    "--sample-articles", 3,
    "--output", SAMPLE_REPORT,
]
run_cli(compare_cmd)
display(Markdown(SAMPLE_REPORT.read_text(encoding="utf-8")))


$ /usr/bin/python3 -m src.embed.compare_embedding_inputs --inputs /content/Text-Mining---RAG-on-News/src/embed/output/chunks/vieonline_news_chunks_token.jsonl /content/Text-Mining---RAG-on-News/src/embed/output/chunks/vieonline_news_chunks_llamaindex.jsonl /content/Text-Mining---RAG-on-News/src/embed/output/chunks/vieonline_news_chunks_structured.jsonl /content/Text-Mining---RAG-on-News/src/embed/output/chunks/vieonline_news_chunks_langchain_recursive.jsonl --sample-articles 3 --output /content/Text-Mining---RAG-on-News/src/embed/output/embedding_strategy_samples.md

{
  "output": "/content/Text-Mining---RAG-on-News/src/embed/output/embedding_strategy_samples.md"
}

[exit=0] elapsed=6.84s


# Embedding Strategy Samples

Report này so sánh text/metadata được đưa vào embedding giữa các chunking strategy cho cùng article.

## Article `10000`

### `token`

- Source: `/content/Text-Mining---RAG-on-News/src/embed/output/chunks/vieonline_news_chunks_token.jsonl`
- Num chunks: 3

| Chunk | Implementation | Structure | Tokens | Preview |
| ---: | --- | --- | ---: | --- |
| 0 | `internal_token_window` | `content` | 400 | Tiêu đề: Tom Cruise vẫn 'đặt cược' sinh mạng trong Mission: Impossible 8 Mô tả: Siêu sao hành động Tom Cruise vẫn miệt mài 'vào sinh ra tử' cùng vai diễn, suýt chết nhiều lần khi đóng phim Mission: Impossible - The Fi... |
| 1 | `internal_token_window` | `content` | 400 | Tiêu đề: Tom Cruise vẫn 'đặt cược' sinh mạng trong Mission: Impossible 8 Mô tả: Siêu sao hành động Tom Cruise vẫn miệt mài 'vào sinh ra tử' cùng vai diễn, suýt chết nhiều lần khi đóng phim Mission: Impossible - The Fi... |
| 2 | `internal_token_window` | `content` | 82 | Tiêu đề: Tom Cruise vẫn 'đặt cược' sinh mạng trong Mission: Impossible 8 Mô tả: Siêu sao hành động Tom Cruise vẫn miệt mài 'vào sinh ra tử' cùng vai diễn, suýt chết nhiều lần khi đóng phim Mission: Impossible - The Fi... |

### `llamaindex`

- Source: `/content/Text-Mining---RAG-on-News/src/embed/output/chunks/vieonline_news_chunks_llamaindex.jsonl`
- Num chunks: 4

| Chunk | Implementation | Structure | Tokens | Preview |
| ---: | --- | --- | ---: | --- |
| 0 | `llama_index.core.node_parser.SentenceSplitter` | `content` | 169 | Tiêu đề: Tom Cruise vẫn 'đặt cược' sinh mạng trong Mission: Impossible 8 Mô tả: Siêu sao hành động Tom Cruise vẫn miệt mài 'vào sinh ra tử' cùng vai diễn, suýt chết nhiều lần khi đóng phim Mission: Impossible - The Fi... |
| 1 | `llama_index.core.node_parser.SentenceSplitter` | `content` | 176 | Tiêu đề: Tom Cruise vẫn 'đặt cược' sinh mạng trong Mission: Impossible 8 Mô tả: Siêu sao hành động Tom Cruise vẫn miệt mài 'vào sinh ra tử' cùng vai diễn, suýt chết nhiều lần khi đóng phim Mission: Impossible - The Fi... |
| 2 | `llama_index.core.node_parser.SentenceSplitter` | `content` | 180 | Tiêu đề: Tom Cruise vẫn 'đặt cược' sinh mạng trong Mission: Impossible 8 Mô tả: Siêu sao hành động Tom Cruise vẫn miệt mài 'vào sinh ra tử' cùng vai diễn, suýt chết nhiều lần khi đóng phim Mission: Impossible - The Fi... |
| 3 | `llama_index.core.node_parser.SentenceSplitter` | `content` | 245 | Tiêu đề: Tom Cruise vẫn 'đặt cược' sinh mạng trong Mission: Impossible 8 Mô tả: Siêu sao hành động Tom Cruise vẫn miệt mài 'vào sinh ra tử' cùng vai diễn, suýt chết nhiều lần khi đóng phim Mission: Impossible - The Fi... |

### `structured`

- Source: `/content/Text-Mining---RAG-on-News/src/embed/output/chunks/vieonline_news_chunks_structured.jsonl`
- Num chunks: 3

| Chunk | Implementation | Structure | Tokens | Preview |
| ---: | --- | --- | ---: | --- |
| 0 | `internal_structured_sentence_window` | `content` | 369 | Tiêu đề: Tom Cruise vẫn 'đặt cược' sinh mạng trong Mission: Impossible 8 Mô tả: Siêu sao hành động Tom Cruise vẫn miệt mài 'vào sinh ra tử' cùng vai diễn, suýt chết nhiều lần khi đóng phim Mission: Impossible - The Fi... |
| 1 | `internal_structured_sentence_window` | `content` | 360 | Tiêu đề: Tom Cruise vẫn 'đặt cược' sinh mạng trong Mission: Impossible 8 Mô tả: Siêu sao hành động Tom Cruise vẫn miệt mài 'vào sinh ra tử' cùng vai diễn, suýt chết nhiều lần khi đóng phim Mission: Impossible - The Fi... |
| 2 | `internal_structured_sentence_window` | `content` | 245 | Tiêu đề: Tom Cruise vẫn 'đặt cược' sinh mạng trong Mission: Impossible 8 Mô tả: Siêu sao hành động Tom Cruise vẫn miệt mài 'vào sinh ra tử' cùng vai diễn, suýt chết nhiều lần khi đóng phim Mission: Impossible - The Fi... |

### `langchain_recursive`

- Source: `/content/Text-Mining---RAG-on-News/src/embed/output/chunks/vieonline_news_chunks_langchain_recursive.jsonl`
- Num chunks: 13

| Chunk | Implementation | Structure | Tokens | Preview |
| ---: | --- | --- | ---: | --- |
| 0 | `langchain_text_splitters.RecursiveCharacterTextSplitter` | `content` | 73 | Tiêu đề: Tom Cruise vẫn 'đặt cược' sinh mạng trong Mission: Impossible 8 Mô tả: Siêu sao hành động Tom Cruise vẫn miệt mài 'vào sinh ra tử' cùng vai diễn, suýt chết nhiều lần khi đóng phim Mission: Impossible - The Fi... |
| 1 | `langchain_text_splitters.RecursiveCharacterTextSplitter` | `content` | 24 | Tiêu đề: Tom Cruise vẫn 'đặt cược' sinh mạng trong Mission: Impossible 8 Mô tả: Siêu sao hành động Tom Cruise vẫn miệt mài 'vào sinh ra tử' cùng vai diễn, suýt chết nhiều lần khi đóng phim Mission: Impossible - The Fi... |
| 2 | `langchain_text_splitters.RecursiveCharacterTextSplitter` | `content` | 61 | Tiêu đề: Tom Cruise vẫn 'đặt cược' sinh mạng trong Mission: Impossible 8 Mô tả: Siêu sao hành động Tom Cruise vẫn miệt mài 'vào sinh ra tử' cùng vai diễn, suýt chết nhiều lần khi đóng phim Mission: Impossible - The Fi... |
| 3 | `langchain_text_splitters.RecursiveCharacterTextSplitter` | `content` | 56 | Tiêu đề: Tom Cruise vẫn 'đặt cược' sinh mạng trong Mission: Impossible 8 Mô tả: Siêu sao hành động Tom Cruise vẫn miệt mài 'vào sinh ra tử' cùng vai diễn, suýt chết nhiều lần khi đóng phim Mission: Impossible - The Fi... |
| 4 | `langchain_text_splitters.RecursiveCharacterTextSplitter` | `content` | 46 | Tiêu đề: Tom Cruise vẫn 'đặt cược' sinh mạng trong Mission: Impossible 8 Mô tả: Siêu sao hành động Tom Cruise vẫn miệt mài 'vào sinh ra tử' cùng vai diễn, suýt chết nhiều lần khi đóng phim Mission: Impossible - The Fi... |

## Article `100056`

### `token`

- Source: `/content/Text-Mining---RAG-on-News/src/embed/output/chunks/vieonline_news_chunks_token.jsonl`
- Num chunks: 5

| Chunk | Implementation | Structure | Tokens | Preview |
| ---: | --- | --- | ---: | --- |
| 0 | `internal_token_window` | `content` | 400 | Tiêu đề: Sắp xếp, sáp nhập trường ĐH: Quyền lợi của người lao động, sinh viên ra sao? Mô tả: Theo các chuyên gia, nhiều vấn đề đặt ra cần giải quyết khi xây dựng đề án sáp nhập các trường đại học như: nhân sự, cơ sở v... |
| 1 | `internal_token_window` | `content` | 400 | Tiêu đề: Sắp xếp, sáp nhập trường ĐH: Quyền lợi của người lao động, sinh viên ra sao? Mô tả: Theo các chuyên gia, nhiều vấn đề đặt ra cần giải quyết khi xây dựng đề án sáp nhập các trường đại học như: nhân sự, cơ sở v... |
| 2 | `internal_token_window` | `content` | 400 | Tiêu đề: Sắp xếp, sáp nhập trường ĐH: Quyền lợi của người lao động, sinh viên ra sao? Mô tả: Theo các chuyên gia, nhiều vấn đề đặt ra cần giải quyết khi xây dựng đề án sáp nhập các trường đại học như: nhân sự, cơ sở v... |
| 3 | `internal_token_window` | `content` | 400 | Tiêu đề: Sắp xếp, sáp nhập trường ĐH: Quyền lợi của người lao động, sinh viên ra sao? Mô tả: Theo các chuyên gia, nhiều vấn đề đặt ra cần giải quyết khi xây dựng đề án sáp nhập các trường đại học như: nhân sự, cơ sở v... |
| 4 | `internal_token_window` | `content` | 370 | Tiêu đề: Sắp xếp, sáp nhập trường ĐH: Quyền lợi của người lao động, sinh viên ra sao? Mô tả: Theo các chuyên gia, nhiều vấn đề đặt ra cần giải quyết khi xây dựng đề án sáp nhập các trường đại học như: nhân sự, cơ sở v... |

### `llamaindex`

- Source: `/content/Text-Mining---RAG-on-News/src/embed/output/chunks/vieonline_news_chunks_llamaindex.jsonl`
- Num chunks: 11

| Chunk | Implementation | Structure | Tokens | Preview |
| ---: | --- | --- | ---: | --- |
| 0 | `llama_index.core.node_parser.SentenceSplitter` | `content` | 154 | Tiêu đề: Sắp xếp, sáp nhập trường ĐH: Quyền lợi của người lao động, sinh viên ra sao? Mô tả: Theo các chuyên gia, nhiều vấn đề đặt ra cần giải quyết khi xây dựng đề án sáp nhập các trường đại học như: nhân sự, cơ sở v... |
| 1 | `llama_index.core.node_parser.SentenceSplitter` | `content` | 166 | Tiêu đề: Sắp xếp, sáp nhập trường ĐH: Quyền lợi của người lao động, sinh viên ra sao? Mô tả: Theo các chuyên gia, nhiều vấn đề đặt ra cần giải quyết khi xây dựng đề án sáp nhập các trường đại học như: nhân sự, cơ sở v... |
| 2 | `llama_index.core.node_parser.SentenceSplitter` | `content` | 181 | Tiêu đề: Sắp xếp, sáp nhập trường ĐH: Quyền lợi của người lao động, sinh viên ra sao? Mô tả: Theo các chuyên gia, nhiều vấn đề đặt ra cần giải quyết khi xây dựng đề án sáp nhập các trường đại học như: nhân sự, cơ sở v... |
| 3 | `llama_index.core.node_parser.SentenceSplitter` | `content` | 151 | Tiêu đề: Sắp xếp, sáp nhập trường ĐH: Quyền lợi của người lao động, sinh viên ra sao? Mô tả: Theo các chuyên gia, nhiều vấn đề đặt ra cần giải quyết khi xây dựng đề án sáp nhập các trường đại học như: nhân sự, cơ sở v... |
| 4 | `llama_index.core.node_parser.SentenceSplitter` | `content` | 180 | Tiêu đề: Sắp xếp, sáp nhập trường ĐH: Quyền lợi của người lao động, sinh viên ra sao? Mô tả: Theo các chuyên gia, nhiều vấn đề đặt ra cần giải quyết khi xây dựng đề án sáp nhập các trường đại học như: nhân sự, cơ sở v... |

### `structured`

- Source: `/content/Text-Mining---RAG-on-News/src/embed/output/chunks/vieonline_news_chunks_structured.jsonl`
- Num chunks: 6

| Chunk | Implementation | Structure | Tokens | Preview |
| ---: | --- | --- | ---: | --- |
| 0 | `internal_structured_sentence_window` | `content` | 379 | Tiêu đề: Sắp xếp, sáp nhập trường ĐH: Quyền lợi của người lao động, sinh viên ra sao? Mô tả: Theo các chuyên gia, nhiều vấn đề đặt ra cần giải quyết khi xây dựng đề án sáp nhập các trường đại học như: nhân sự, cơ sở v... |
| 1 | `internal_structured_sentence_window` | `content` | 381 | Tiêu đề: Sắp xếp, sáp nhập trường ĐH: Quyền lợi của người lao động, sinh viên ra sao? Mô tả: Theo các chuyên gia, nhiều vấn đề đặt ra cần giải quyết khi xây dựng đề án sáp nhập các trường đại học như: nhân sự, cơ sở v... |
| 2 | `internal_structured_sentence_window` | `content` | 386 | Tiêu đề: Sắp xếp, sáp nhập trường ĐH: Quyền lợi của người lao động, sinh viên ra sao? Mô tả: Theo các chuyên gia, nhiều vấn đề đặt ra cần giải quyết khi xây dựng đề án sáp nhập các trường đại học như: nhân sự, cơ sở v... |
| 3 | `internal_structured_sentence_window` | `content` | 372 | Tiêu đề: Sắp xếp, sáp nhập trường ĐH: Quyền lợi của người lao động, sinh viên ra sao? Mô tả: Theo các chuyên gia, nhiều vấn đề đặt ra cần giải quyết khi xây dựng đề án sáp nhập các trường đại học như: nhân sự, cơ sở v... |
| 4 | `internal_structured_sentence_window` | `content` | 388 | Tiêu đề: Sắp xếp, sáp nhập trường ĐH: Quyền lợi của người lao động, sinh viên ra sao? Mô tả: Theo các chuyên gia, nhiều vấn đề đặt ra cần giải quyết khi xây dựng đề án sáp nhập các trường đại học như: nhân sự, cơ sở v... |

### `langchain_recursive`

- Source: `/content/Text-Mining---RAG-on-News/src/embed/output/chunks/vieonline_news_chunks_langchain_recursive.jsonl`
- Num chunks: 24

| Chunk | Implementation | Structure | Tokens | Preview |
| ---: | --- | --- | ---: | --- |
| 0 | `langchain_text_splitters.RecursiveCharacterTextSplitter` | `content` | 65 | Tiêu đề: Sắp xếp, sáp nhập trường ĐH: Quyền lợi của người lao động, sinh viên ra sao? Mô tả: Theo các chuyên gia, nhiều vấn đề đặt ra cần giải quyết khi xây dựng đề án sáp nhập các trường đại học như: nhân sự, cơ sở v... |
| 1 | `langchain_text_splitters.RecursiveCharacterTextSplitter` | `content` | 63 | Tiêu đề: Sắp xếp, sáp nhập trường ĐH: Quyền lợi của người lao động, sinh viên ra sao? Mô tả: Theo các chuyên gia, nhiều vấn đề đặt ra cần giải quyết khi xây dựng đề án sáp nhập các trường đại học như: nhân sự, cơ sở v... |
| 2 | `langchain_text_splitters.RecursiveCharacterTextSplitter` | `content` | 47 | Tiêu đề: Sắp xếp, sáp nhập trường ĐH: Quyền lợi của người lao động, sinh viên ra sao? Mô tả: Theo các chuyên gia, nhiều vấn đề đặt ra cần giải quyết khi xây dựng đề án sáp nhập các trường đại học như: nhân sự, cơ sở v... |
| 3 | `langchain_text_splitters.RecursiveCharacterTextSplitter` | `content` | 65 | Tiêu đề: Sắp xếp, sáp nhập trường ĐH: Quyền lợi của người lao động, sinh viên ra sao? Mô tả: Theo các chuyên gia, nhiều vấn đề đặt ra cần giải quyết khi xây dựng đề án sáp nhập các trường đại học như: nhân sự, cơ sở v... |
| 4 | `langchain_text_splitters.RecursiveCharacterTextSplitter` | `content` | 74 | Tiêu đề: Sắp xếp, sáp nhập trường ĐH: Quyền lợi của người lao động, sinh viên ra sao? Mô tả: Theo các chuyên gia, nhiều vấn đề đặt ra cần giải quyết khi xây dựng đề án sáp nhập các trường đại học như: nhân sự, cơ sở v... |

## Article `10007`

### `token`

- Source: `/content/Text-Mining---RAG-on-News/src/embed/output/chunks/vieonline_news_chunks_token.jsonl`
- Num chunks: 2

| Chunk | Implementation | Structure | Tokens | Preview |
| ---: | --- | --- | ---: | --- |
| 0 | `internal_token_window` | `content` | 400 | Tiêu đề: Bandai quyết tâm đưa Gundam lên màn ảnh rộng Mô tả: Sau nhiều năm gián đoạn, dự án phim Gundam live-action cuối cùng cũng chính thức sản xuất. Chuyên mục: Giải trí Đoạn nội dung: Những phác thảo ban đầu của d... |
| 1 | `internal_token_window` | `content` | 249 | Tiêu đề: Bandai quyết tâm đưa Gundam lên màn ảnh rộng Mô tả: Sau nhiều năm gián đoạn, dự án phim Gundam live-action cuối cùng cũng chính thức sản xuất. Chuyên mục: Giải trí Đoạn nội dung: tại Bắc Mỹ thông qua anime, p... |

### `llamaindex`

- Source: `/content/Text-Mining---RAG-on-News/src/embed/output/chunks/vieonline_news_chunks_llamaindex.jsonl`
- Num chunks: 4

| Chunk | Implementation | Structure | Tokens | Preview |
| ---: | --- | --- | ---: | --- |
| 0 | `llama_index.core.node_parser.SentenceSplitter` | `content` | 166 | Tiêu đề: Bandai quyết tâm đưa Gundam lên màn ảnh rộng Mô tả: Sau nhiều năm gián đoạn, dự án phim Gundam live-action cuối cùng cũng chính thức sản xuất. Chuyên mục: Giải trí Đoạn nội dung: Những phác thảo ban đầu của d... |
| 1 | `llama_index.core.node_parser.SentenceSplitter` | `content` | 121 | Tiêu đề: Bandai quyết tâm đưa Gundam lên màn ảnh rộng Mô tả: Sau nhiều năm gián đoạn, dự án phim Gundam live-action cuối cùng cũng chính thức sản xuất. Chuyên mục: Giải trí Đoạn nội dung: Universal Century là điểm khở... |
| 2 | `llama_index.core.node_parser.SentenceSplitter` | `content` | 175 | Tiêu đề: Bandai quyết tâm đưa Gundam lên màn ảnh rộng Mô tả: Sau nhiều năm gián đoạn, dự án phim Gundam live-action cuối cùng cũng chính thức sản xuất. Chuyên mục: Giải trí Đoạn nội dung: Legendary Entertainmentcũng t... |
| 3 | `llama_index.core.node_parser.SentenceSplitter` | `content` | 140 | Tiêu đề: Bandai quyết tâm đưa Gundam lên màn ảnh rộng Mô tả: Sau nhiều năm gián đoạn, dự án phim Gundam live-action cuối cùng cũng chính thức sản xuất. Chuyên mục: Giải trí Đoạn nội dung: Tất cả đều khởi đầu từ animeM... |

### `structured`

- Source: `/content/Text-Mining---RAG-on-News/src/embed/output/chunks/vieonline_news_chunks_structured.jsonl`
- Num chunks: 2

| Chunk | Implementation | Structure | Tokens | Preview |
| ---: | --- | --- | ---: | --- |
| 0 | `internal_structured_sentence_window` | `content` | 385 | Tiêu đề: Bandai quyết tâm đưa Gundam lên màn ảnh rộng Mô tả: Sau nhiều năm gián đoạn, dự án phim Gundam live-action cuối cùng cũng chính thức sản xuất. Chuyên mục: Giải trí Đoạn nội dung: Những phác thảo ban đầu của d... |
| 1 | `internal_structured_sentence_window` | `content` | 283 | Tiêu đề: Bandai quyết tâm đưa Gundam lên màn ảnh rộng Mô tả: Sau nhiều năm gián đoạn, dự án phim Gundam live-action cuối cùng cũng chính thức sản xuất. Chuyên mục: Giải trí Đoạn nội dung: "Ông lớn" của ngành giải trí... |

### `langchain_recursive`

- Source: `/content/Text-Mining---RAG-on-News/src/embed/output/chunks/vieonline_news_chunks_langchain_recursive.jsonl`
- Num chunks: 9

| Chunk | Implementation | Structure | Tokens | Preview |
| ---: | --- | --- | ---: | --- |
| 0 | `langchain_text_splitters.RecursiveCharacterTextSplitter` | `content` | 64 | Tiêu đề: Bandai quyết tâm đưa Gundam lên màn ảnh rộng Mô tả: Sau nhiều năm gián đoạn, dự án phim Gundam live-action cuối cùng cũng chính thức sản xuất. Chuyên mục: Giải trí Đoạn nội dung: Những phác thảo ban đầu của d... |
| 1 | `langchain_text_splitters.RecursiveCharacterTextSplitter` | `content` | 56 | Tiêu đề: Bandai quyết tâm đưa Gundam lên màn ảnh rộng Mô tả: Sau nhiều năm gián đoạn, dự án phim Gundam live-action cuối cùng cũng chính thức sản xuất. Chuyên mục: Giải trí Đoạn nội dung: . Đây là một trong những sự k... |
| 2 | `langchain_text_splitters.RecursiveCharacterTextSplitter` | `content` | 48 | Tiêu đề: Bandai quyết tâm đưa Gundam lên màn ảnh rộng Mô tả: Sau nhiều năm gián đoạn, dự án phim Gundam live-action cuối cùng cũng chính thức sản xuất. Chuyên mục: Giải trí Đoạn nội dung: . Tuy chưa có nhiều chi tiết... |
| 3 | `langchain_text_splitters.RecursiveCharacterTextSplitter` | `content` | 63 | Tiêu đề: Bandai quyết tâm đưa Gundam lên màn ảnh rộng Mô tả: Sau nhiều năm gián đoạn, dự án phim Gundam live-action cuối cùng cũng chính thức sản xuất. Chuyên mục: Giải trí Đoạn nội dung: . Universal Century là điểm k... |
| 4 | `langchain_text_splitters.RecursiveCharacterTextSplitter` | `content` | 60 | Tiêu đề: Bandai quyết tâm đưa Gundam lên màn ảnh rộng Mô tả: Sau nhiều năm gián đoạn, dự án phim Gundam live-action cuối cùng cũng chính thức sản xuất. Chuyên mục: Giải trí Đoạn nội dung: . Tuy nhiên, với thông báo mớ... |



## 6. Embed từng strategy và quan sát tiến độ

Cell này là bước chậm nhất trên CPU. `tqdm` sẽ hiển thị tiến độ theo batch. Nếu quá chậm, giảm `ARTICLE_LIMIT`, giảm số `STRATEGIES`, hoặc giữ `BATCH_SIZE=4`.

In [11]:
for strategy, chunk_file in zip(STRATEGIES, chunk_files):
    output_dir = DENSE_OUTPUT_ROOT / strategy
    embed_cmd = [
        sys.executable, "-m", "src.embed.embed_chunks",
        "--input", chunk_file,
        "--output-dir", output_dir,
        "--model", MODEL_NAME,
        "--batch-size", BATCH_SIZE,
        "--sample-size", SAMPLE_SIZE,
        "--show-samples",
    ]
    run_cli(embed_cmd)


$ /usr/bin/python3 -m src.embed.embed_chunks --input /content/Text-Mining---RAG-on-News/src/embed/output/chunks/vieonline_news_chunks_token.jsonl --output-dir /content/Text-Mining---RAG-on-News/src/embed/output/dense/token --model intfloat/multilingual-e5-large --batch-size 4 --sample-size 5 --show-samples


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 21883.23it/s]

Embedding: 100%|██████████| 6025/6025 [40:29<00:00,  2.48batch/s]
{
  "stats": {
    "chunks_read": 24099,
    "chunks_embedded": 24099,
    "skipped_empty_text": 0,
    "embedding_dimension": 1024,
    "batch_size": 4,
    "elapsed_seconds": 2429.915227,
    "chunks_per_second": 9.9176,
    "token_stats": {
      "min": 94,
      "max": 533,
      "avg": 397.959
    },
    "char_stats": {
      "min": 478,
      "max": 3099,
      "avg": 1831.9081
    },
    "strategy_counts": {
      "token": 24099
    },
    "implementation_counts": {
      "internal_token_window": 23803,
      "single_small_article": 296
  

## 7. Bảng thống kê embedding

In [12]:
def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

rows = []
for strategy in STRATEGIES:
    strategy_dir = DENSE_OUTPUT_ROOT / strategy
    stats_path = strategy_dir / "embedding_stats.json"
    manifest_path = strategy_dir / "manifest.json"
    if not stats_path.exists():
        print(f"Missing stats for {strategy}: {stats_path}")
        continue
    stats = load_json(stats_path)
    manifest = load_json(manifest_path)
    rows.append({
        "strategy": strategy,
        "chunks": stats["chunks_embedded"],
        "dimension": stats["embedding_dimension"],
        "batch_size": stats["batch_size"],
        "elapsed_seconds": stats["elapsed_seconds"],
        "chunks_per_second": stats["chunks_per_second"],
        "avg_tokens": stats["token_stats"]["avg"],
        "max_tokens": stats["token_stats"]["max"],
        "avg_chars": stats["char_stats"]["avg"],
        "max_chars": stats["char_stats"]["max"],
        "embeddings_path": manifest["embeddings_path"],
        "stats_path": str(stats_path),
    })

embed_stats_df = pd.DataFrame(rows).sort_values("strategy")
display(embed_stats_df)

summary_csv = PROJECT_ROOT / "src" / "embed" / "output" / "embedding_stats_summary.csv"
embed_stats_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
print("Saved:", summary_csv)

,strategy,chunks,dimension,batch_size,elapsed_seconds,chunks_per_second,avg_tokens,max_tokens,avg_chars,max_chars,embeddings_path,stats_path
3,langchain_recursive,98156,1024,4,4261.312797,23.0342,139.0185,303,639.4621,1381,/content/Text-Mining---RAG-on-News/src/embed/o...,/content/Text-Mining---RAG-on-News/src/embed/o...
1,llamaindex,48977,1024,4,2916.000963,16.7959,219.1943,362,1010.7891,1666,/content/Text-Mining---RAG-on-News/src/embed/o...,/content/Text-Mining---RAG-on-News/src/embed/o...
2,structured,25464,1024,4,2577.720623,9.8785,396.7930,587,1819.2870,3213,/content/Text-Mining---RAG-on-News/src/embed/o...,/content/Text-Mining---RAG-on-News/src/embed/o...
0,token,24099,1024,4,2429.915227,9.9176,397.9590,533,1831.9081,3099,/content/Text-Mining---RAG-on-News/src/embed/o...,/content/Text-Mining---RAG-on-News/src/embed/o...


Saved: /content/Text-Mining---RAG-on-News/src/embed/output/embedding_stats_summary.csv


## 8. Debug chunk dài nhất và mẫu đã embed

In [13]:
for strategy in STRATEGIES:
    strategy_dir = DENSE_OUTPUT_ROOT / strategy
    stats_path = strategy_dir / "embedding_stats.json"
    samples_path = strategy_dir / "debug_samples.jsonl"
    if not stats_path.exists():
        continue
    stats = load_json(stats_path)
    display(Markdown(f"### {strategy}: longest chunks"))
    display(pd.DataFrame(stats["longest_chunks"]))
    if samples_path.exists():
        samples = [json.loads(line) for line in samples_path.read_text(encoding="utf-8").splitlines() if line.strip()]
        display(Markdown(f"### {strategy}: debug samples"))
        display(pd.DataFrame(samples))

### token: longest chunks

,chunk_id,article_id,strategy,estimated_tokens,text_chars,title,category
0,75615_token_0000,75615,token,533,2441,"Tới Breda, Hà Lan thăm lâu đài trên mặt nước v...",Du lịch
1,77248_token_0000,77248,token,532,2460,"Du lịch Phú Quốc, bạn đã biết các địa điểm này...",Du lịch
2,77124_token_0000,77124,token,531,2416,Những địa điểm hấp dẫn khi đến Lyon du khách t...,Du lịch
3,77535_token_0000,77535,token,531,2375,5 món đặc sản ở Vĩnh Hy không thể bỏ lỡ trong ...,Du lịch
4,76610_token_0000,76610,token,530,2356,Lý Sơn không chỉ có tỏi ngon mà còn 'quá trời'...,Du lịch
5,77750_token_0000,77750,token,520,2450,'Điểm danh' những kinh đô thời trang hàng đầu ...,Du lịch
6,75902_token_0000,75902,token,519,2389,Tham khảo 5 điểm dừng nghỉ được yêu thích tại ...,Du lịch
7,76062_token_0000,76062,token,518,2383,Kinh nghiệm cắm trại hồ Trị An 'cực chill' cho...,Du lịch
8,77657_token_0000,77657,token,518,2336,Khám phá Buôn Ma Thuột: Thủ phủ cà phê đậm văn...,Du lịch
9,110946_token_0000,110946,token,517,2309,Người tiêu dùng Việt Nam dẫn đầu khu vực về mứ...,Kinh doanh


### token: debug samples

,chunk_id,article_id,strategy,implementation,chunk_index,title,category,estimated_tokens,text_chars,embedding_input_preview,chunk_text_preview
0,211640_token_0000,211640,token,internal_token_window,0,"Không muốn hại thận, cần hạn chế 4 loại thịt",Sức khỏe,452,2122,"passage: Tiêu đề: Không muốn hại thận, cần hạn...","Theo Sohu, nhiều người khi nhắc đến tăng axit ..."
1,211640_token_0001,211640,token,internal_token_window,1,"Không muốn hại thận, cần hạn chế 4 loại thịt",Sức khỏe,319,1490,"passage: Tiêu đề: Không muốn hại thận, cần hạn...","đặc biệt khi kết hợp với lẩu hay canh hầm, có ..."
2,152685_token_0000,152685,token,internal_token_window,0,Bất ngờ danh tính doanh nghiệp trúng thầu khu ...,Bất động sản,471,2141,passage: Tiêu đề: Bất ngờ danh tính doanh nghi...,"Ngày 11/1, thông tin từ Sở TN-MT tỉnh Thừa Thi..."
3,152685_token_0001,152685,token,internal_token_window,1,Bất ngờ danh tính doanh nghiệp trúng thầu khu ...,Bất động sản,173,809,passage: Tiêu đề: Bất ngờ danh tính doanh nghi...,"khu đất nói trên được công bố, nhiều người tỏ ..."
4,132354_token_0000,132354,token,internal_token_window,0,Bí thư Tỉnh ủy Khánh Hòa: 'An toàn của người d...,Thời sự,467,2110,passage: Tiêu đề: Bí thư Tỉnh ủy Khánh Hòa: 'A...,"Chiều và tối 6.11, ông Nghiêm Xuân Thành, Bí t..."


### llamaindex: longest chunks

...

,chunk_id,article_id,strategy,implementation,chunk_index,title,category,estimated_tokens,text_chars,embedding_input_preview,chunk_text_preview
0,211640_structured_0000,211640,structured,internal_structured_sentence_window,0,"Không muốn hại thận, cần hạn chế 4 loại thịt",Sức khỏe,441,2072,"passage: Tiêu đề: Không muốn hại thận, cần hạn...","Theo Sohu, nhiều người khi nhắc đến tăng axit ..."
1,211640_structured_0001,211640,structured,internal_structured_sentence_window,1,"Không muốn hại thận, cần hạn chế 4 loại thịt",Sức khỏe,347,1620,"passage: Tiêu đề: Không muốn hại thận, cần hạn...","Nội tạng động vật như gan, thận, lòng, dạ dày ..."
2,152685_structured_0000,152685,structured,internal_structured_sentence_window,0,Bất ngờ danh tính doanh nghiệp trúng thầu khu ...,Bất động sản,465,2111,passage: Tiêu đề: Bất ngờ danh tính doanh nghi...,"Ngày 11/1, thông tin từ Sở TN-MT tỉnh Thừa Thi..."
3,152685_structured_0001,152685,structured,internal_structured_sentence_window,1,Bất ngờ danh tính doanh nghiệp trúng thầu khu ...,Bất động sản,181,843,passage: Tiêu đề: Bất ngờ danh tính doanh nghi...,Sau khi kết quả trúng đấu giá của khu đất nói ...
4,132354_structured_0000,132354,structured,internal_structured_sentence_window,0,Bí thư Tỉnh ủy Khánh Hòa: 'An toàn của người d...,Thời sự,447,2009,passage: Tiêu đề: Bí thư Tỉnh ủy Khánh Hòa: 'A...,"Chiều và tối 6. 11, ông Nghiêm Xuân Thành, Bí ..."


### langchain_recursive: longest chunks

,chunk_id,article_id,strategy,estimated_tokens,text_chars,title,category
0,108978_langchain_recursive_0000,108978,langchain_recursive,303,1381,"Chuyển đổi số trong thu thập, giám sát thông s...",Kinh doanh
1,101967_langchain_recursive_0000,101967,langchain_recursive,289,1272,Điểm thi lớp 10 của TP.HCM: Môn toán có 36 thí...,Giáo dục
2,152719_langchain_recursive_0000,152719,langchain_recursive,287,1257,Về Nam Ô nghe giấc mơ làng biển,Bất động sản
3,45089_langchain_recursive_0000,45089,langchain_recursive,287,1255,Có tiền sao không xài?,Thời sự
4,208438_langchain_recursive_0000,208438,langchain_recursive,284,1267,Docquity Việt Nam tài trợ hội nghị Sản Phụ kho...,Sức khỏe
5,239260_langchain_recursive_0000,239260,langchain_recursive,283,1279,"Va chạm trên cao tốc: Một ô tô lật ngang, một ...",Thời sự
6,121007_langchain_recursive_0000,121007,langchain_recursive,283,1308,Hai tàu ngầm Nga bắn trúng mục tiêu mồi ở vùng...,Thế giới
7,203640_langchain_recursive_0000,203640,langchain_recursive,281,1245,Bắt Chi cục trưởng Thi hành án dân sự ở Bến Tr...,Pháp luật
8,246677_langchain_recursive_0000,246677,langchain_recursive,280,1278,Phá đường dây ma túy từ nước ngoài về Việt Nam...,Thời sự
9,216863_langchain_recursive_0000,216863,langchain_recursive,278,1294,Ukraine bắn hạ máy bay không người lái mới siê...,Thế giới


### langchain_recursive: debug samples

,chunk_id,article_id,strategy,implementation,chunk_index,title,category,estimated_tokens,text_chars,embedding_input_preview,chunk_text_preview
0,211640_langchain_recursive_0000,211640,langchain_recursive,langchain_text_splitters.RecursiveCharacterTex...,0,"Không muốn hại thận, cần hạn chế 4 loại thịt",Sức khỏe,135,630,"passage: Tiêu đề: Không muốn hại thận, cần hạn...","Theo Sohu, nhiều người khi nhắc đến tăng axit ..."
1,211640_langchain_recursive_0001,211640,langchain_recursive,langchain_text_splitters.RecursiveCharacterTex...,1,"Không muốn hại thận, cần hạn chế 4 loại thịt",Sức khỏe,131,616,"passage: Tiêu đề: Không muốn hại thận, cần hạn...",. Thịt dê được xếp vào nhóm thịt đỏ nên purin ...
2,211640_langchain_recursive_0002,211640,langchain_recursive,langchain_text_splitters.RecursiveCharacterTex...,2,"Không muốn hại thận, cần hạn chế 4 loại thịt",Sức khỏe,128,605,"passage: Tiêu đề: Không muốn hại thận, cần hạn...",". Ngoài thịt dê, nhiều loại thịt đỏ khác như t..."
3,211640_langchain_recursive_0003,211640,langchain_recursive,langchain_text_splitters.RecursiveCharacterTex...,3,"Không muốn hại thận, cần hạn chế 4 loại thịt",Sức khỏe,123,575,"passage: Tiêu đề: Không muốn hại thận, cần hạn...",. Trong nhóm người có axit uric cao kèm giảm c...
4,211640_langchain_recursive_0004,211640,langchain_recursive,langchain_text_splitters.RecursiveCharacterTex...,4,"Không muốn hại thận, cần hạn chế 4 loại thịt",Sức khỏe,126,604,"passage: Tiêu đề: Không muốn hại thận, cần hạn...",". Nội tạng động vật như gan, thận, lòng, dạ dà..."


## 9. Search thử trên một strategy đã embed

In [14]:
QUERY = "Tin tức về công nghệ AI tại Việt Nam"
SEARCH_STRATEGY = STRATEGIES[0]

search_cmd = [
    sys.executable, "-m", "src.embed.dense_search",
    "--index-dir", DENSE_OUTPUT_ROOT / SEARCH_STRATEGY,
    "--query", QUERY,
    "--top-k", 5,
    "--model", MODEL_NAME,
]
run_cli(search_cmd)


$ /usr/bin/python3 -m src.embed.dense_search --index-dir /content/Text-Mining---RAG-on-News/src/embed/output/dense/token --query Tin tức về công nghệ AI tại Việt Nam --top-k 5 --model intfloat/multilingual-e5-large


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6106.77it/s]
{
  "query": "Tin tức về công nghệ AI tại Việt Nam",
  "results": [
    {
      "chunk_id": "65416_token_0002",
      "article_id": "65416",
      "score": 0.8897868990898132,
      "title": "AI Việt góp phần định hình bối cảnh trí tuệ nhân tạo Đông Nam Á",
      "category": "Khoa học công nghệ",
      "strategy": "token",
      "chunk_index": 2,
      "text": "Tiêu đề: AI Việt góp phần định hình bối cảnh trí tuệ nhân tạo Đông Nam Á\nMô tả: Với rào cản kỹ thuật được hạ thấp, các startup trí tuệ nhân tạo ở Đông Nam Á, trong đó có công ty AI Việt Nam có thể tập trung vào thế mạnh của mình là giải quyết vấn đề kinh doanh thực tế .\nChuyên mục: Khoa học công nghệ\nĐoạn nội dung:\ncác doanh nghiệp mạnh hơn đế

'\nLoading weights:   0%|          | 0/391 [00:00<?, ?it/s]\nLoading weights: 100%|██████████| 391/391 [00:00<00:00, 6106.77it/s]\n{\n  "query": "Tin tức về công nghệ AI tại Việt Nam",\n  "results": [\n    {\n      "chunk_id": "65416_token_0002",\n      "article_id": "65416",\n      "score": 0.8897868990898132,\n      "title": "AI Việt góp phần định hình bối cảnh trí tuệ nhân tạo Đông Nam Á",\n      "category": "Khoa học công nghệ",\n      "strategy": "token",\n      "chunk_index": 2,\n      "text": "Tiêu đề: AI Việt góp phần định hình bối cảnh trí tuệ nhân tạo Đông Nam Á\\nMô tả: Với rào cản kỹ thuật được hạ thấp, các startup trí tuệ nhân tạo ở Đông Nam Á, trong đó có công ty AI Việt Nam có thể tập trung vào thế mạnh của mình là giải quyết vấn đề kinh doanh thực tế .\\nChuyên mục: Khoa học công nghệ\\nĐoạn nội dung:\\ncác doanh nghiệp mạnh hơn đến từ Mỹ, Trung Quốc... chưa có được. AI Hay hiện được cung cấp thông qua website và ứng dụng cùng tên trên các kho phần mềm của hai nền tảng 

## 10. Eval retrieval trên 50 QA mẫu

Cell này đánh giá các dense index đã embed bằng qrels ở mức `article_id`. Mỗi retrieved chunk được xem là đúng nếu `article_id` của chunk nằm trong `source_article_ids` hoặc `article_id` của QA.

Metric chính để xếp hạng là `nDCG@10`. Các metric khác dùng để giải thích trade-off chất lượng/chi phí.

In [19]:
QA_PATH = "Dataset/QA_Claude/QA_output.csv"
QA_SAMPLE_SIZE = 153
EVAL_TOP_K = 10

qa_df = pd.read_csv(QA_PATH)
# if "is_possible" in qa_df.columns:
#     qa_df = qa_df[qa_df["is_possible"].astype(str).str.lower().isin(["true", "1", "yes"])]
# qa_df = qa_df.dropna(subset=["question"]).head(QA_SAMPLE_SIZE).copy()

print("QA samples:", len(qa_df))
display(qa_df[[col for col in ["id", "article_id", "question", "source_article_ids", "qa_type"] if col in qa_df.columns]].head())

QA samples: 152


,id,article_id,question,source_article_ids,qa_type
0,211640_1,211640,Những loại nội tạng động vật nào được khuyến c...,NaN,factoid
1,211640_2,211640,Quan niệm cho rằng thịt dê là thủ phạm duy nhấ...,NaN,event_summary
2,211640_3,211640,Tại sao nước dùng hầm xương hoặc nước lẩu lại ...,NaN,cause_effect
3,211640_4,211640,Hàm lượng purin trong cá nhỏ và nội tạng động ...,NaN,comparison
4,211640_5,211640,Đúng hay sai: 'Chỉ cần bỏ thịt dê và chuyển sa...,NaN,claim_verification


In [20]:
import math
import statistics
import numpy as np
from sentence_transformers import SentenceTransformer

from src.embed.embed_chunks import prepare_query_text
from src.embed.dense_search import load_index


def parse_qrels(row):
    values = set()
    for col in ["source_article_ids", "article_id"]:
        if col not in row or pd.isna(row[col]):
            continue
        raw = row[col]
        if isinstance(raw, (list, tuple, set)):
            parts = raw
        else:
            text = str(raw).strip()
            try:
                parsed = json.loads(text)
                parts = parsed if isinstance(parsed, list) else [parsed]
            except Exception:
                parts = text.replace(";", ",").replace("|", ",").split(",")
        for part in parts:
            value = str(part).strip().strip("[]'\"")
            if value and value.lower() not in {"nan", "none", ""}:
                values.add(value)
    return values


def dcg_at_k(relevances, k):
    return sum(rel / math.log2(idx + 2) for idx, rel in enumerate(relevances[:k]))


def metrics_for_results(results, relevant_article_ids, k=10):
    retrieved_article_ids = []
    seen_article_ids = set()
    for item in results:
        article_id = str(item.get("article_id"))
        if article_id in seen_article_ids:
            continue
        seen_article_ids.add(article_id)
        retrieved_article_ids.append(article_id)
        if len(retrieved_article_ids) >= k:
            break

    relevances = [1 if article_id in relevant_article_ids else 0 for article_id in retrieved_article_ids]
    ideal_relevances = [1] * min(len(relevant_article_ids), k)
    ideal_dcg = dcg_at_k(ideal_relevances, k)
    ndcg = dcg_at_k(relevances, k) / ideal_dcg if ideal_dcg else 0.0

    def recall_at(cutoff):
        found = set(retrieved_article_ids[:cutoff]) & relevant_article_ids
        return len(found) / len(relevant_article_ids) if relevant_article_ids else 0.0

    def hit_at(cutoff):
        return 1.0 if set(retrieved_article_ids[:cutoff]) & relevant_article_ids else 0.0

    mrr = 0.0
    for idx, article_id in enumerate(retrieved_article_ids[:k], start=1):
        if article_id in relevant_article_ids:
            mrr = 1.0 / idx
            break

    return {
        "nDCG@10": ndcg,
        "Recall@5": recall_at(5),
        "Recall@10": recall_at(10),
        "MRR@10": mrr,
        "Hit@1": hit_at(1),
        "Hit@5": hit_at(5),
    }


def percentile(values, pct):
    if not values:
        return 0.0
    values = sorted(values)
    index = min(len(values) - 1, math.ceil((pct / 100) * len(values)) - 1)
    return values[index]


def directory_size_mb(path):
    path = Path(path)
    return sum(file.stat().st_size for file in path.rglob("*") if file.is_file()) / (1024 * 1024)


def search_arrays(query, embeddings, metadata, model, top_k):
    query_vector = model.encode([prepare_query_text(query)], normalize_embeddings=True, show_progress_bar=False)
    query_array = np.asarray(query_vector, dtype=np.float32)[0]
    scores = embeddings @ query_array
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [metadata[int(index)] | {"score": float(scores[int(index)])} for index in top_indices]

In [21]:
# Load model một lần để query evaluation không tải lại model theo từng query.
query_model = SentenceTransformer(MODEL_NAME)

eval_rows = []
per_query_rows = []

for strategy in STRATEGIES:
    index_dir = DENSE_OUTPUT_ROOT / strategy
    stats_path = index_dir / "embedding_stats.json"
    manifest_path = index_dir / "manifest.json"
    if not (index_dir / "embeddings.npy").exists():
        print(f"Skip {strategy}: missing embeddings.npy")
        continue

    embeddings, metadata = load_index(index_dir)
    stats = load_json(stats_path)
    manifest = load_json(manifest_path)
    query_latencies_ms = []
    metric_rows = []

    for _, qa in qa_df.iterrows():
        relevant_article_ids = parse_qrels(qa)
        if not relevant_article_ids:
            continue
        started = time.perf_counter()
        results = search_arrays(str(qa["question"]), embeddings, metadata, query_model, EVAL_TOP_K)
        latency_ms = (time.perf_counter() - started) * 1000
        query_latencies_ms.append(latency_ms)

        metrics = metrics_for_results(results, relevant_article_ids, k=EVAL_TOP_K)
        metric_rows.append(metrics)
        per_query_rows.append({
            "strategy": strategy,
            "qa_id": qa.get("id"),
            "qa_type": qa.get("qa_type"),
            "question": qa.get("question"),
            "relevant_article_ids": sorted(relevant_article_ids),
            "top_article_ids": [item.get("article_id") for item in results[:EVAL_TOP_K]],
            "latency_ms": latency_ms,
            **metrics,
        })

    if not metric_rows:
        continue

    aggregate = {key: float(np.mean([row[key] for row in metric_rows])) for key in metric_rows[0]}
    eval_rows.append({
        "Model": MODEL_NAME,
        "Chunking": strategy,
        "nDCG@10": aggregate["nDCG@10"],
        "Recall@5": aggregate["Recall@5"],
        "Recall@10": aggregate["Recall@10"],
        "MRR@10": aggregate["MRR@10"],
        "Hit@1": aggregate["Hit@1"],
        "Hit@5": aggregate["Hit@5"],
        "embedding_time_seconds": stats["elapsed_seconds"],
        "chunks_per_second": stats["chunks_per_second"],
        "query_latency_ms_avg": statistics.mean(query_latencies_ms),
        "query_latency_ms_p95": percentile(query_latencies_ms, 95),
        "index_size_mb": directory_size_mb(index_dir),
        "embedding_dimension": manifest["embedding_dimension"],
        "num_chunks": stats["chunks_embedded"],
        "avg_chunk_tokens": stats["token_stats"]["avg"],
        "queries": len(metric_rows),
    })

eval_df = pd.DataFrame(eval_rows)
if not eval_df.empty:
    eval_df = eval_df.sort_values(
        ["nDCG@10", "Recall@10", "query_latency_ms_avg"],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    eval_df.insert(0, "Rank", range(1, len(eval_df) + 1))

display(eval_df)

per_query_df = pd.DataFrame(per_query_rows)
eval_output_dir = PROJECT_ROOT / "src" / "embed" / "output" / "eval"
eval_output_dir.mkdir(parents=True, exist_ok=True)
eval_df.to_csv(eval_output_dir / "leaderboard_qa50.csv", index=False, encoding="utf-8-sig")
per_query_df.to_json(eval_output_dir / "per_query_results_qa50.jsonl", orient="records", lines=True, force_ascii=False)
print("Saved leaderboard:", eval_output_dir / "leaderboard_qa50.csv")
print("Saved per-query results:", eval_output_dir / "per_query_results_qa50.jsonl")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

,Rank,Model,Chunking,nDCG@10,Recall@5,Recall@10,MRR@10,Hit@1,Hit@5,embedding_time_seconds,chunks_per_second,query_latency_ms_avg,query_latency_ms_p95,index_size_mb,embedding_dimension,num_chunks,avg_chunk_tokens,queries
0,1,intfloat/multilingual-e5-large,token,0.788094,0.817982,0.837719,0.836842,0.763158,0.934211,2429.915227,9.9176,33.543827,40.145281,212.347048,1024,24099,397.9590,152
1,2,intfloat/multilingual-e5-large,structured,0.778245,0.817982,0.835526,0.825987,0.736842,0.934211,2577.720623,9.8785,33.488932,40.626828,224.141803,1024,25464,396.7930,152
2,3,intfloat/multilingual-e5-large,llamaindex,0.755538,0.780702,0.789474,0.817928,0.736842,0.907895,2916.000963,16.7959,51.272990,83.414858,332.691929,1024,48977,219.1943,152
3,4,intfloat/multilingual-e5-large,langchain_recursive,0.702540,0.721491,0.721491,0.782018,0.710526,0.861842,4261.312797,23.0342,68.850295,100.475502,578.294096,1024,98156,139.0185,152


Saved leaderboard: /content/Text-Mining---RAG-on-News/src/embed/output/eval/leaderboard_qa50.csv
Saved per-query results: /content/Text-Mining---RAG-on-News/src/embed/output/eval/per_query_results_qa50.jsonl


## 11. Notes đánh giá

Quy tắc đọc bảng:

- Metric xếp hạng chính: `nDCG@10`.
- Nếu `nDCG@10` gần nhau, ưu tiên `Recall@10` cao hơn và `query_latency_ms_avg` thấp hơn.
- Không so sánh các dòng nếu dùng khác query set hoặc khác qrels.
- Với E5, notebook dùng prefix `query: ` cho câu hỏi và `passage: ` cho chunk trong bước embed.
- Kết quả này là eval nhanh trên khoảng 50 QA mẫu, chưa thay thế evaluation đầy đủ cho báo cáo cuối.

Các câu hỏi cần trả lời từ leaderboard:

1. Strategy chunking nào ổn định nhất?
2. Model/chunking nào đạt retrieval tốt nhất theo `nDCG@10`?
3. Cấu hình nào trade-off tốt giữa chất lượng, latency và index size?
4. Cấu hình nào nên dùng cho pipeline RAG cuối cùng?